In [ ]:
# ==============================================================================
# 🚀 DO NOT MODIFY: Standardized Notebook Setup
# ==============================================================================
# This cell is designed to work in both Google Colab and local environments.
# It ensures that the environment is correctly configured by cloning (or
# locating) the project repository and installing the necessary dependencies.
#
# ------------------------------------------------------------------------------
#
#  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):
#
#  This cell will automatically find the repository root and configure your
#  environment. Just make sure you have run: pip install -e .[dev]
#
# ------------------------------------------------------------------------------

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# --- Configuration ---
REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"
REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned
# --- End of Configuration ---


def find_repo_root(start_path: Path) -> Path | None:
    """
    Find the repository root by looking for pyproject.toml.

    Searches upward from start_path until it finds pyproject.toml or hits root.

    Args:
        start_path: Directory to start searching from.

    Returns:
        Path to repository root, or None if not found.
    """
    current = start_path.resolve()
    while current != current.parent:  # Stop at filesystem root
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None


def detect_active_branch(repo_dir: Path) -> str:
    """
    Determine the active git branch for pulling updates.

    Tries multiple methods to detect the current branch name.

    Args:
        repo_dir: Path to the git repository.

    Returns:
        Branch name (defaults to 'master' if detection fails).
    """
    commands = [
        "git symbolic-ref --short HEAD",
        "git rev-parse --abbrev-ref HEAD",
    ]
    for cmd in commands:
        result = subprocess.run(
            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True
        )
        if result.returncode == 0:
            branch = result.stdout.strip()
            if branch and not branch.startswith("origin/"):
                return branch
    return "master"


def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:
    """
    Run a shell command and raise an error if it fails.

    Args:
        cmd: The command to run.
        cwd: Optional working directory for the command.

    Raises:
        RuntimeError: If the command returns a non-zero exit code.
    """
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


def load_setup_module(repo_path: Path):
    """Load the setup module directly without triggering package imports."""
    setup_path = repo_path / "core" / "notebook" / "setup.py"
    spec = importlib.util.spec_from_file_location("_setup_module", setup_path)
    setup_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(setup_module)
    return setup_module


# --- Detect environment ---
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --- Main setup logic ---
if IN_COLAB:
    print("☁️  Running in Google Colab. Setting up the environment...\n")

    # Determine repository path
    start_dir = Path.cwd()
    if start_dir.name == REPO_DIR.name:
        repo_path = start_dir
    else:
        repo_path = start_dir / REPO_DIR

    # Clone or update repository
    if not repo_path.exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")
        print(f"✅ Repository cloned to {repo_path}\n")
    else:
        print(f"📂 Repository already exists at {repo_path}")
        active_branch = detect_active_branch(repo_path)
        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")
        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)
        print(f"✅ Repository updated\n")

    # Verify repository structure
    if not (repo_path / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "
            "The repository may be corrupted."
        )

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    # Install dependencies (smart installation - only installs missing packages)
    # Load the setup module directly to avoid triggering other package imports
    setup = load_setup_module(repo_path)

    result = setup.smart_install_dependencies(
        repo_path=repo_path,
        include_dev=False,
        verbose=True,
    )

    # Fail loudly if critical packages failed to install
    if result["failed"]:
        print(f"\n⚠️  WARNING: {len(result['failed'])} packages failed to install:")
        for pkg in result["failed"]:
            print(f"  - {pkg}")
        print("\nYou may encounter import errors. Please check your internet connection.")

    print("\n" + "=" * 70)
    print("✅ Environment setup complete! You can now proceed with the notebook.")
    print("=" * 70)

else:
    print("💻 Running in local environment. Configuring...\n")

    # Find the repository root
    repo_path = find_repo_root(Path.cwd())

    if repo_path is None:
        raise FileNotFoundError(
            "Could not find repository root (no pyproject.toml found). "
            "Please ensure you are running this notebook from within the "
            "ADH-LLM-Tutorials-2025 repository directory."
        )

    print(f"✅ Found repository root: {repo_path}")

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)

    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    print("\n" + "=" * 70)
    print("✅ Local environment configured successfully!")
    print("=" * 70)
    print("\n⚠️  Please ensure you have run: pip install -e .[dev]")
    print("(Required for local development)")

# 09 - Interacting with a Local LLM via an OpenAI-Compatible API

## Introduction

In this notebook, you'll learn how to interact with Large Language Models (LLMs) using the **OpenAI API format**, which has become the industry standard for LLM interactions. We'll use a local Ollama server that provides an OpenAI-compatible API, allowing us to use the official `openai` Python library.

### Why This Matters

The OpenAI API is the *de facto* standard for working with LLMs. By learning this API contract, you'll be able to:
- Work with any LLM provider (OpenAI, Anthropic, local models, etc.) using the same interface
- Structure inputs and outputs in a consistent, predictable way
- Build reliable applications that can switch between different models seamlessly

### Learning Objectives

By the end of this notebook, you will:
1. **Generate basic text responses** from an LLM using the standard `messages` format
2. **Request structured JSON output** and validate it with Pydantic for reliability
3. **Implement function calling** to enable the LLM to use Python functions as tools

Let's begin!

In [ ]:
# Import required libraries
import json

from openai import OpenAI
from pydantic import BaseModel

from core.llm import setup_ollama_llm
from core.llm.tools import calculate_bmi, get_drug_interaction

In [ ]:
# Set up the local LLM via Ollama
print("Setting up local Ollama LLM...\n")
model_name = setup_ollama_llm()

# Create an OpenAI client pointing to the local Ollama server
client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama's OpenAI-compatible endpoint
    api_key="ollama",  # Required by the OpenAI client, but not used by Ollama
)

print("\n✅ Local LLM is ready!")
print(f"Model: {model_name}")
print("\nThe OpenAI client is configured to use the local Ollama server.")

In [ ]:
# Quick verification: Make a simple test call
print("Testing the model with a simple query...")

test_response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "Say 'Hello from the Ollama backend!'"}],
    max_tokens=20,
)

print(f"Response: {test_response.choices[0].message.content}")

## Part 1: Basic Text Generation

The foundation of all LLM interactions is the **messages array**. This is a list of dictionaries, where each dictionary represents a message with a `role` (either "system", "user", or "assistant") and `content` (the actual text).

### The Messages Format

- **system**: Sets the behavior and context for the AI (e.g., "You are a helpful medical assistant")
- **user**: Represents input from the human user
- **assistant**: Represents previous responses from the AI (used for multi-turn conversations)

Let's start with a simple example:

In [ ]:
# Create a messages array
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful medical education assistant. "
            "Provide clear, concise explanations."
        ),
    },
    {
        "role": "user",
        "content": (
            "What is the difference between Type 1 and Type 2 Diabetes? "
            "Answer in 2-3 sentences."
        ),
    },
]

# Call the LLM
print("Calling the LLM...")
response = client.chat.completions.create(model=model_name, messages=messages)

# Extract and print the response
answer = response.choices[0].message.content
print("\n" + "=" * 80)
print("LLM Response:")
print("=" * 80)
print(answer)

### What Just Happened?

1. We defined a `messages` array with a system prompt and a user question
2. We called `client.chat.completions.create()` which:
   - Sent our messages to the local Ollama server
   - Processed them through the Llama 3.2 model
   - Returned a structured response object
3. We extracted the text from `response.choices[0].message.content`

This is the fundamental pattern for all LLM interactions!

## Part 2: Structured Output with JSON Mode

While free-form text is useful, many applications need **structured, reliable data** from LLMs. For example:
- Extracting patient information from clinical notes
- Parsing medication lists into standardized formats
- Converting unstructured reports into database records

We can use the OpenAI client's **structured output** feature with Pydantic models.

### Step 1: Define the Expected Structure

First, we create a Pydantic model that defines the schema we want:

In [ ]:
class PatientSummary(BaseModel):
    """Structured representation of key patient information."""

    patient_id: str
    age: int
    primary_diagnosis: str
    medications: list[str]


# Display the schema (this helps us understand what we're asking the LLM to produce)
print("Expected JSON Schema:")
print(json.dumps(PatientSummary.model_json_schema(), indent=2))

### Step 2: Request Structured Output from the LLM

Now we'll provide a clinical note and ask the LLM to extract information into our defined schema:

In [ ]:
# Create a prompt that includes the clinical note
clinical_note = """
Patient ID: MRN-78492
Age: 67
Chief Complaint: Shortness of breath and fatigue

Assessment: Patient presents with symptoms consistent with congestive heart failure.
Recent echocardiogram shows reduced ejection fraction of 35%.

Current Medications:
- Lisinopril 10mg daily
- Metoprolol 50mg twice daily
- Furosemide 40mg daily
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are a medical data extraction assistant. "
            "Extract information accurately from clinical notes."
        ),
    },
    {
        "role": "user",
        "content": (
            "Extract the patient_id, age, primary_diagnosis, and medications "
            f"from this clinical note:\n\n{clinical_note}"
        ),
    },
]

# Call the LLM with structured output
print("Requesting structured JSON output...")
response = client.beta.chat.completions.parse(
    model=model_name,
    messages=messages,
    response_format=PatientSummary,
)

# The response is automatically parsed into a Pydantic model
patient_summary = response.choices[0].message.parsed

print("\n" + "=" * 80)
print("Validated Patient Summary:")
print("=" * 80)
print(f"Patient ID: {patient_summary.patient_id}")
print(f"Age: {patient_summary.age}")
print(f"Primary Diagnosis: {patient_summary.primary_diagnosis}")
print(f"Medications: {', '.join(patient_summary.medications)}")

### What Just Happened?

1. We created a Pydantic model (`PatientSummary`) defining our expected structure
2. We used `client.beta.chat.completions.parse()` with `response_format=PatientSummary`
3. The OpenAI client automatically:
   - Sent the schema to the model
   - Requested JSON output
   - Parsed and validated the response into a Pydantic model

**Key Insight:** This pattern ensures that your application receives **reliable, type-safe data** from the LLM, not just free-form text that needs manual parsing.

## Part 3: Tool Use (Function Calling)

The most powerful LLM pattern is **function calling** (also called tool use). This allows the LLM to:
1. Recognize when it needs external information or computation
2. Request the execution of a specific Python function with specific arguments
3. Use the function's return value to generate a final answer

This is how ChatGPT can search the web, perform calculations, or query databases!

### The Two-Step Flow

```
User Question → LLM → "I need to call calculate_bmi(90, 1.8)" → 
Execute Function → Return Result → LLM → Final Natural Language Answer
```

Let's implement this step by step.

### Step 1: Define Tool Schemas

First, we need to describe our Python functions to the LLM in a format it understands. This is the OpenAI function schema format:

In [ ]:
# Define the tool schemas (these describe our Python functions to the LLM)
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_bmi",
            "description": (
                "Calculates the Body Mass Index (BMI) given "
                "weight in kilograms and height in meters."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "weight_kg": {
                        "type": "number",
                        "description": "The patient's weight in kilograms.",
                    },
                    "height_m": {
                        "type": "number",
                        "description": "The patient's height in meters.",
                    },
                },
                "required": ["weight_kg", "height_m"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_drug_interaction",
            "description": "Checks for known interactions between two drugs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "drug_a": {
                        "type": "string",
                        "description": "The name of the first drug.",
                    },
                    "drug_b": {
                        "type": "string",
                        "description": "The name of the second drug.",
                    },
                },
                "required": ["drug_a", "drug_b"],
            },
        },
    },
]

print("Tool schemas defined:")
for tool in tools:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

### Step 2: First API Call - LLM Requests a Tool

Now we'll ask a question that requires a calculation. Watch how the LLM recognizes it needs to use the `calculate_bmi` function:

In [ ]:
# Create a user question that requires a tool
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful medical assistant with access to tools for "
            "calculations and drug interactions."
        ),
    },
    {
        "role": "user",
        "content": (
            "What is the BMI for a patient who weighs 90 kg and is 1.8 meters "
            "tall? Please interpret the result."
        ),
    },
]

# Call the LLM with tools available
print("First API Call - LLM decides what to do...")
response = client.chat.completions.create(
    model=model_name, messages=messages, tools=tools, tool_choice="auto"
)

# Extract the assistant's message
assistant_message = response.choices[0].message

print("\n" + "=" * 80)
print("LLM Response:")
print("=" * 80)

# Check if the LLM wants to call a tool
if assistant_message.tool_calls:
    print("✅ The LLM has requested a tool call!\n")
    for tool_call in assistant_message.tool_calls:
        print(f"Function: {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")
        print(f"Tool Call ID: {tool_call.id}")
else:
    print("The LLM provided a direct answer:")
    print(assistant_message.content)

### Step 3: Execute the Tool

Now we (the client application) need to actually run the Python function the LLM requested:

In [ ]:
# Parse the tool call and execute the function
if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[
        0
    ]  # Get the first (and likely only) tool call

    # Parse the arguments (they're returned as a JSON string)
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    print("Executing the function on the client side...")
    print(f"Function: {function_name}")
    print(f"Arguments: {function_args}")

    # Map function names to actual Python functions
    available_functions = {
        "calculate_bmi": calculate_bmi,
        "get_drug_interaction": get_drug_interaction,
    }

    # Call the actual function
    function_to_call = available_functions[function_name]
    try:
        function_result = function_to_call(**function_args)
    except ValueError as exc:
        function_result = f"Tool execution failed: {exc}"
        print(f"⚠️ {function_result}")
    else:
        print(f"\nFunction returned: {function_result}")
        print(f"Type: {type(function_result)}")

### Step 4: Second API Call - LLM Uses the Result

Finally, we send the function's result back to the LLM so it can generate a natural language answer:

In [ ]:
if assistant_message.tool_calls:
    # Add the assistant's tool call message to the conversation
    messages.append(assistant_message)

    # Add the tool result message
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": str(function_result),  # The function's return value
        }
    )

    # Make the second API call
    print("\nSecond API Call - LLM generates final answer...")
    final_response = client.chat.completions.create(
        model=model_name, messages=messages, tools=tools
    )

    # Extract the final answer
    final_answer = final_response.choices[0].message.content

    print("\n" + "=" * 80)
    print("Final LLM Response:")
    print("=" * 80)
    print(final_answer)

### What Just Happened?

This is the complete function calling workflow:

1. **First API Call**: 
   - We sent the user's question with `tools` parameter
   - The LLM recognized it needed to calculate BMI
   - It responded with a `tool_call` object (not a text answer)

2. **Client-Side Execution**:
   - We parsed the `tool_call` to extract function name and arguments
   - We called the actual Python function `calculate_bmi(90.0, 1.8)`
   - We should have got back the result: `27.78`

3. **Second API Call**:
   - We added the function result to the conversation as a `tool` role message
   - The LLM now had the calculation result
   - It generated a natural language answer interpreting the BMI value

This pattern allows LLMs to **extend their capabilities** beyond just text generation!

What do you notice about the BMI? Does it match our expectations? Does the model's interpretation make sense? How can we fix this?

## Challenge: Try the Drug Interaction Tool

Now it's your turn! Modify the code above to:
1. Ask a question that requires the `get_drug_interaction` function
2. Observe the LLM requesting the tool
3. Execute the function
4. See the LLM's final interpretation

**Hint**: Try asking about the interaction between "warfarin" and "aspirin" (a known dangerous combination in our toy database).

### Solution Cell (Try on Your Own First!)

In [ ]:
# Solution: Drug interaction query
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful medical assistant with access to tools for "
            "calculations and drug interactions."
        ),
    },
    {
        "role": "user",
        "content": "Is it safe to prescribe warfarin and aspirin together?",
    },
]

# First call
response = client.chat.completions.create(
    model=model_name, messages=messages, tools=tools, tool_choice="auto"
)
assistant_message = response.choices[0].message

if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    print(f"LLM called: {function_name}({function_args})")

    # Execute function
    available_functions = {
        "calculate_bmi": calculate_bmi,
        "get_drug_interaction": get_drug_interaction,
    }
    try:
        function_result = available_functions[function_name](**function_args)
    except ValueError as exc:
        function_result = f"Tool execution failed: {exc}"
        print(f"⚠️ {function_result}")
    else:
        print(f"Result: {function_result}")

    # Second call
    messages.append(assistant_message)
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": str(function_result),
        }
    )

    final_response = client.chat.completions.create(
        model=model_name, messages=messages, tools=tools
    )
    print("\nFinal Answer:")
    print(final_response.choices[0].message.content)

## Summary: Core LLM Interaction Patterns

Congratulations! You've learned the three fundamental patterns for working with LLMs:

1. **Basic Generation**: Using the `messages` array to send prompts and receive text
2. **Structured Output**: Requesting JSON responses and validating them with Pydantic
3. **Function Calling**: Enabling LLMs to use external tools and data sources

### Key Takeaways

- The **OpenAI API format** is the industry standard, making your code portable across providers
- **Local Ollama servers** can provide OpenAI-compatible APIs, allowing you to use the official `openai` client
- **Structured output** makes LLM responses reliable enough for production applications
- **Function calling** is the bridge between LLMs and the real world (databases, APIs, calculations)

Now let's put these concepts into practice with hands-on extension exercises!

---

## Extension Exercises

Ready to apply what you've learned? These exercises will help you build your own structured extractors and custom tools for real-world clinical scenarios.

### Exercise 1: Build Your Own Structured Data Extractor
### Exercise 2: Create Your Own Clinical Tool

Let's dive in!

## Extension Exercise 1: Build Your Own Structured Data Extractor

In Part 2, you saw how to extract patient information using a predefined `PatientSummary` schema. Now it's your turn to create a custom extractor for a different clinical scenario!

### The Challenge

You'll build a structured data extractor for **vital signs documentation**. Nurses and physicians often record vital signs in free-text notes, but for trend analysis and clinical decision support, this data needs to be in a structured format.

### Step 1: Define Your Pydantic Schema

Create a Pydantic model that captures vital signs data. Your schema should include:
- **Temperature** (float, in Celsius)
- **Blood Pressure** (systolic and diastolic as separate integers)
- **Heart Rate** (integer, beats per minute)
- **Respiratory Rate** (integer, breaths per minute)
- **Oxygen Saturation** (integer, percentage)

**Try it yourself** in the cell below before looking at the solution!

In [ ]:
# TODO: Define your VitalSigns Pydantic model here
# Hint: Use the PatientSummary model from Part 2 as a reference
# Remember to include type hints for each field!

# class VitalSigns(BaseModel):
#     """Structured representation of patient vital signs."""
#     # Your fields here...
#     pass

### Step 2: Sample Clinical Note

Here's a realistic nursing note with vital signs embedded in free text:

In [ ]:
vital_signs_note = """
Nursing Assessment - 14:30

Patient appears comfortable and in no acute distress. Vital signs obtained:
Temperature 37.2°C (oral), blood pressure 142/88 mmHg, pulse 78 bpm and regular.
Respiratory rate counted at 16 breaths per minute, oxygen saturation 96% on room air.

Patient denies chest pain, shortness of breath, or palpitations. 
Lungs clear to auscultation bilaterally.
"""

print("Sample Clinical Note:")
print("=" * 80)
print(vital_signs_note)

### Step 3: Extract the Data

Now use your `VitalSigns` model to extract structured data from the note. Fill in the TODOs below:

In [ ]:
# TODO: Complete this code to extract vital signs

# messages = [
#     {
#         "role": "system",
#         "content": "You are a medical data extraction assistant. Extract vital signs accurately."
#     },
#     {
#         "role": "user",
#         "content": f"Extract all vital signs from this note:\n\n{vital_signs_note}"
#     }
# ]

# # TODO: Call client.beta.chat.completions.parse() with your VitalSigns model
# # response = ...

# # TODO: Extract the parsed result
# # vital_signs = ...

# # TODO: Print the structured data
# print("Extracted Vital Signs:")
# print(f"Temperature: {vital_signs.temperature}°C")
# # ... print the rest of the fields

### Solution (Try on your own first!)

Expand the cell below to see a complete working solution:

In [ ]:
# SOLUTION - Exercise 1: Vital Signs Extraction


class VitalSigns(BaseModel):
    """Structured representation of patient vital signs."""

    temperature_celsius: float
    systolic_bp: int
    diastolic_bp: int
    heart_rate: int
    respiratory_rate: int
    oxygen_saturation: int


# Create the extraction prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are a medical data extraction assistant. "
            "Extract vital signs accurately from clinical notes."
        ),
    },
    {
        "role": "user",
        "content": f"Extract all vital signs from this nursing note:\n\n{vital_signs_note}",
    },
]

# Call the LLM with structured output
print("Extracting structured vital signs data...")
response = client.beta.chat.completions.parse(
    model=model_name, messages=messages, response_format=VitalSigns
)

# Get the validated Pydantic model
vital_signs = response.choices[0].message.parsed

# Display the extracted data
print("\n" + "=" * 80)
print("✅ Extracted Vital Signs (Validated & Structured):")
print("=" * 80)
print(f"Temperature: {vital_signs.temperature_celsius}°C")
print(f"Blood Pressure: {vital_signs.systolic_bp}/{vital_signs.diastolic_bp} mmHg")
print(f"Heart Rate: {vital_signs.heart_rate} bpm")
print(f"Respiratory Rate: {vital_signs.respiratory_rate} breaths/min")
print(f"Oxygen Saturation: {vital_signs.oxygen_saturation}%")

print("\n" + "=" * 80)
print("Key Benefits:")
print("=" * 80)
print("• Data is now in a format ready for database storage")
print("• Type validation ensures data integrity (no strings where numbers should be)")
print("• Can easily trend vitals over time or trigger alerts")
print("• Eliminates manual data entry errors")

---

## Extension Exercise 2: Create Your Own Clinical Tool

In Part 3, you used pre-built tools (`calculate_bmi` and `get_drug_interaction`). Now you'll create your own clinical calculation tool from scratch!

### The Challenge

You'll implement the **CHADS2-VASc score calculator**, which is used to assess stroke risk in patients with atrial fibrillation. This is a real clinical tool used daily in cardiology.

The CHADS2-VASc score assigns points for various risk factors:
- **C**ongestive heart failure: 1 point
- **H**ypertension: 1 point
- **A**ge ≥75: 2 points
- **D**iabetes: 1 point
- **S**troke/TIA history: 2 points
- **V**ascular disease: 1 point
- **A**ge 65-74: 1 point
- **S**ex category (female): 1 point

**Score interpretation:**
- 0: Low risk (no anticoagulation)
- 1: Consider anticoagulation
- ≥2: Anticoagulation recommended

### Step 1: Implement the Python Function

Create the calculation function with proper type hints and validation:

In [ ]:
# TODO: Implement the CHADS2-VASc calculator

# def calculate_chads2_vasc_score(
#     age: int,
#     sex: str,  # "male" or "female"
#     has_chf: bool,
#     has_hypertension: bool,
#     has_stroke_tia: bool,
#     has_vascular_disease: bool,
#     has_diabetes: bool,
# ) -> dict[str, int | str]:
#     """
#     Calculate CHADS2-VASc score for stroke risk in atrial fibrillation.
#
#     Args:
#         age: Patient age in years
#         sex: Patient sex ("male" or "female")
#         has_chf: Congestive heart failure history
#         has_hypertension: Hypertension diagnosis
#         has_stroke_tia: Prior stroke or TIA
#         has_vascular_disease: Vascular disease history
#         has_diabetes: Diabetes diagnosis
#
#     Returns:
#         Dictionary with "score" and "risk_category"
#     """
#     # TODO: Implement the scoring logic
#     # Remember: Age ≥75 = 2 points, Age 65-74 = 1 point
#     # Female sex = 1 point
#     # Each condition = specified points per guidelines
#     pass

### Step 2: Define the Tool Schema

Create the OpenAI function schema that describes your tool to the LLM:

In [ ]:
# TODO: Define the tool schema for CHADS2-VASc calculator

# chads2_vasc_tool = {
#     "type": "function",
#     "function": {
#         "name": "calculate_chads2_vasc_score",
#         "description": "Calculate CHADS2-VASc stroke risk score for patients with atrial fibrillation",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 # TODO: Define properties for each parameter
#                 # Use the calculate_bmi tool schema as a reference
#             },
#             "required": [
#                 # TODO: List all required parameters
#             ],
#         },
#     },
# }

### Step 3: Test Your Tool

Use the complete function calling workflow to test your tool:

In [ ]:
# TODO: Test your CHADS2-VASc calculator with a clinical scenario

# test_query = """
# I have a 72-year-old female patient with atrial fibrillation.
# She has a history of hypertension and diabetes, but no prior stroke.
# Should she be on anticoagulation?
# """

# # TODO: Create messages array
# # TODO: First API call with your tool
# # TODO: Execute the function
# # TODO: Second API call with the result
# # TODO: Display the final answer

### Complete Solution (Try on your own first!)

Below is a full working implementation of the CHADS2-VASc calculator with the complete tool calling workflow:

In [ ]:
# SOLUTION - Exercise 2: CHADS2-VASc Calculator


# Step 1: Implement the function
def calculate_chads2_vasc_score(
    age: int,
    sex: str,
    has_chf: bool,
    has_hypertension: bool,
    has_stroke_tia: bool,
    has_vascular_disease: bool,
    has_diabetes: bool,
) -> dict[str, int | str]:
    """
    Calculate CHADS2-VASc score for stroke risk in atrial fibrillation.

    Args:
        age: Patient age in years
        sex: Patient sex ("male" or "female")
        has_chf: Congestive heart failure history
        has_hypertension: Hypertension diagnosis
        has_stroke_tia: Prior stroke or TIA
        has_vascular_disease: Vascular disease history
        has_diabetes: Diabetes diagnosis

    Returns:
        Dictionary with "score" and "risk_category"

    Raises:
        ValueError: If age is invalid or sex is not "male" or "female"
    """
    # Validate inputs
    if age < 0 or age > 120:
        raise ValueError(f"Invalid age: {age}. Age must be between 0 and 120.")

    if sex.lower() not in ["male", "female"]:
        raise ValueError(f"Invalid sex: {sex}. Must be 'male' or 'female'.")

    # Calculate score
    score = 0

    # Age points
    if age >= 75:
        score += 2
    elif age >= 65:
        score += 1

    # Sex points (female = 1)
    if sex.lower() == "female":
        score += 1

    # Condition points
    if has_chf:
        score += 1
    if has_hypertension:
        score += 1
    if has_stroke_tia:
        score += 2
    if has_vascular_disease:
        score += 1
    if has_diabetes:
        score += 1

    # Determine risk category
    if score == 0:
        risk_category = "Low risk - No anticoagulation recommended"
    elif score == 1:
        risk_category = "Moderate risk - Consider anticoagulation"
    else:
        risk_category = "High risk - Anticoagulation recommended"

    return {"score": score, "risk_category": risk_category}


# Step 2: Define the tool schema
chads2_vasc_tool = {
    "type": "function",
    "function": {
        "name": "calculate_chads2_vasc_score",
        "description": (
            "Calculate CHADS2-VASc stroke risk score for patients with "
            "atrial fibrillation to guide anticoagulation decisions"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "age": {"type": "integer", "description": "Patient age in years"},
                "sex": {
                    "type": "string",
                    "description": "Patient sex: 'male' or 'female'",
                    "enum": ["male", "female"],
                },
                "has_chf": {
                    "type": "boolean",
                    "description": "History of congestive heart failure",
                },
                "has_hypertension": {
                    "type": "boolean",
                    "description": "Diagnosis of hypertension",
                },
                "has_stroke_tia": {
                    "type": "boolean",
                    "description": "Prior stroke or transient ischemic attack",
                },
                "has_vascular_disease": {
                    "type": "boolean",
                    "description": (
                        "History of vascular disease (MI, PAD, aortic plaque)"
                    ),
                },
                "has_diabetes": {
                    "type": "boolean",
                    "description": "Diabetes mellitus diagnosis",
                },
            },
            "required": [
                "age",
                "sex",
                "has_chf",
                "has_hypertension",
                "has_stroke_tia",
                "has_vascular_disease",
                "has_diabetes",
            ],
        },
    },
}

print("✅ CHADS2-VASc calculator and tool schema defined!")
print(f"\nFunction: {chads2_vasc_tool['function']['name']}")
print(f"Description: {chads2_vasc_tool['function']['description']}")

In [ ]:
# Step 3: Test the tool with a clinical scenario

clinical_scenario = """
I have a 72-year-old female patient with atrial fibrillation. 
She has a history of hypertension and diabetes, but no prior stroke, 
heart failure, or vascular disease. 
What is her CHADS2-VASc score and should she be on anticoagulation?
"""

# Create messages with the tool available
messages = [
    {
        "role": "system",
        "content": (
            "You are a cardiology assistant with access to clinical "
            "calculators. Use the CHADS2-VASc calculator to assess "
            "stroke risk in atrial fibrillation patients."
        ),
    },
    {"role": "user", "content": clinical_scenario},
]

# First API call - LLM decides to use the tool
print("Step 1: First API call - LLM analyzes the question...")
print("=" * 80)
response = client.chat.completions.create(
    model=model_name, messages=messages, tools=[chads2_vasc_tool], tool_choice="auto"
)

assistant_message = response.choices[0].message

if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    print(f"✅ LLM requested tool: {function_name}")
    print(f"Arguments extracted from query: {json.dumps(function_args, indent=2)}")

    # Execute the function
    print(f"\nStep 2: Executing {function_name}...")
    print("=" * 80)

    try:
        result = calculate_chads2_vasc_score(**function_args)
        print(f"Result: {result}")
    except ValueError as exc:
        result = f"Error: {exc}"
        print(f"⚠️ {result}")

    # Second API call - LLM uses the result
    print("\nStep 3: Second API call - LLM interprets the result...")
    print("=" * 80)

    messages.append(assistant_message)
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": str(result),
        }
    )

    final_response = client.chat.completions.create(
        model=model_name, messages=messages, tools=[chads2_vasc_tool]
    )

    print("\n" + "=" * 80)
    print("Final Clinical Recommendation:")
    print("=" * 80)
    print(final_response.choices[0].message.content)
else:
    print("LLM provided direct answer without using tool:")
    print(assistant_message.content)

### What You've Accomplished

By completing these exercises, you've learned to:
- ✅ **Design custom Pydantic schemas** for domain-specific data extraction
- ✅ **Implement clinical calculators** with proper validation and error handling
- ✅ **Create OpenAI-compatible tool schemas** from scratch
- ✅ **Orchestrate the complete tool calling workflow** (query → tool call → execution → interpretation)

These are production-ready patterns used in real digital health applications!

---

## Final Conclusion: Building Production LLM Applications

Congratulations on completing this comprehensive tutorial on LLM interactions! You now have the foundational skills to build sophisticated digital health applications.

### What You've Mastered

1. **Basic Text Generation**
   - Structuring prompts with system/user/assistant roles
   - Controlling LLM behavior through system prompts
   - Managing multi-turn conversations

2. **Structured Data Extraction**
   - Defining Pydantic schemas for type-safe data
   - Extracting structured information from clinical notes
   - Validating LLM outputs automatically
   - Building custom extractors for your domain

3. **Tool Use / Function Calling**
   - Implementing Python functions as LLM tools
   - Creating OpenAI-compatible tool schemas
   - Orchestrating the two-step tool calling workflow
   - Building custom clinical calculators (BMI, CHADS2-VASc, etc.)

### Real-World Applications in Digital Health

These patterns enable powerful applications:

**Clinical Decision Support:**
- Extract patient data from notes → Calculate risk scores → Generate recommendations
- Example: Automated sepsis risk assessment from vital signs and lab values

**Medical Documentation:**
- Physician dictation → Structured SOAP note → Auto-populated EHR fields
- Example: Convert free-text visit notes to billing-ready ICD codes

**Patient-Facing Tools:**
- Patient symptoms (natural language) → Structured triage → Clinical pathways
- Example: Chatbot that collects history and routes to appropriate care level

**Research & Analytics:**
- Unstructured clinical notes → Structured database → Population health insights
- Example: Extract all medication mentions from 1M+ notes for drug utilization studies

### Architectural Best Practices

When building production LLM applications:

1. **Always validate LLM outputs**  
   Use Pydantic models, not string parsing. LLMs are probabilistic - enforce contracts.

2. **Design deterministic tools**  
   Your Python functions should be predictable. The LLM provides the intelligence; your tools provide reliability.

3. **Handle errors gracefully**  
   LLMs can hallucinate, tools can fail. Always have fallback strategies and clear error messages for users.

4. **Use the OpenAI standard**  
   Even with local models, use OpenAI-compatible APIs for portability and ecosystem access.

5. **Monitor and log**  
   In production, log all LLM calls (prompts, outputs, tool calls) for debugging and compliance.

### Beyond This Tutorial

You're now equipped to explore:
- **Retrieval-Augmented Generation (RAG):** Combine semantic search (Notebook 08) with LLMs to build question-answering systems over medical literature
- **Fine-tuning:** Adapt pre-trained models to your specific medical domain
- **Multi-agent systems:** Chain multiple LLM calls with different tools for complex workflows
- **Evaluation frameworks:** Systematically test and improve LLM application quality

### Suggested Projects

Try building:
1. **Clinical note summarizer** that extracts vitals, diagnoses, medications, and plan
2. **Drug interaction checker** that queries a database and provides patient-friendly explanations
3. **Symptom triage bot** that uses multiple calculators (NEWS score, qSOFA, etc.) to assess acuity
4. **Medical coding assistant** that suggests ICD-10 codes from clinical documentation

---

### Thank You!

You've completed the LLM module of this Digital Health tutorial series. The skills you've learned here - prompt engineering, structured extraction, and tool orchestration - are the foundation of modern AI-powered healthcare applications.